# Ottawa zoning bylaw: EDA


In [ ]:
from pathlib import Path

import pandas as pd
import pymupdf
from IPython.display import Image, display

In [ ]:
DATA_DIR = Path("../data/ottawa")
BYLAW_PDF = DATA_DIR / "2026-050.pdf"  # rename to match your download

assert BYLAW_PDF.exists(), f"Download the bylaw PDF to {BYLAW_PDF.resolve()}"
pdf = pymupdf.open(BYLAW_PDF)

In [ ]:
toc = pdf.get_toc()  # [[level, title, page_number], ...]
toc_df = pd.DataFrame(toc, columns=["level", "title", "page"])

content = []
for page_num, page in enumerate(pdf.pages()):
    content.append({"page": page_num + 1, "text": page.get_text()})
content = pd.DataFrame.from_records(content)

df = toc_df.merge(content, on="page", how="outer")

image_index = []
# TODO: get alt text on images

for page_num, page in enumerate(pdf.pages()):
    images = {"page": page_num + 1, "filenames": []}

    for _, img in enumerate(page.get_images(full=True)):
        xref = img[0]
        info = pdf.extract_image(xref)
        fname = f"p{page_num + 1}_x{xref}.{info['ext']}"
        path = (DATA_DIR / "figures" / fname).resolve()
        with open(path, "wb") as f:
            f.write(info["image"])  # save to disk
        images["filenames"].append(fname)
    image_index.append(images)
df = df.merge(pd.DataFrame(image_index), on="page", how="left")

In [ ]:
# temporary, remove all page=-1
df = df[df["page"] > 0]

# many figures are missing level and titles, ffill from previous pages
df[["level", "title"]] = df[["level", "title"]].ffill()

# features
df["n_chars"] = df["text"].str.len()

In [ ]:
# TODO: may need to tune snap tolerance
# v1 - keep as big table
# TODO: get annotations
# for page in doc:
#     for annot in page.annots():
#         print(f'Annotation on page: {page.number} with type: {annot.type} and rect: {annot.rect}')

counter = 20
for i, page in enumerate(pdf[41:45]):
    print(f"page {i}")
    tables = page.find_tables(snap_x_tolerance=10)
    for table in tables:
        print(table.header)
        print(table.col_count)
        print(table.row_count)
        print(table.to_markdown())
    if i > counter:
        break


# TODO: need to detect tables that span multiple pages
# if first column has roman numerals, can potentially join,
# will keep detected tables on page as column
# add another column to dataframe of distinct tables which will span multiple pages
# some rows may also span multiple pages
# will need to copy over header, makes assumption that first row is always header


# section 199 definitions, each likely needs to be its own vector

In [ ]:
chunks = pdf.to_chunks()  # ty: ignore[unresolved-attribute]

In [ ]:
for c in chunks[40:45]:
    print(c.metadata)
    print("=======CHUNK")
    print(c.text)

In [ ]:
display(Image(pdf[0].get_pixmap(dpi=100).tobytes("png")))